## Handoff Multi-Agent Design Pattern with MAF and Foundry Agents

![handoff](./Images/handoff.png)

### Installing Utilities and Libraries

In [ ]:
%pip install agent-framework==1.19.0 azure-ai-projects==2.7.0 azure-identity==1.25.3 azure-monitor-opentelemetry==1.8.10

### Setting up the Environment Variables

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import AzureCliCredential

load_dotenv()
foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

credential = AzureCliCredential()

### Creating the Foundry Chat Client

In [ ]:
from agent_framework import Agent
from azure.ai.projects import AIProjectClient
from agent_framework.foundry import FoundryAgent, FoundryChatClient, to_prompt_agent
from azure.identity import AzureCliCredential

client = FoundryChatClient(
    project_endpoint = foundry_project_endpoint,
    model = model_deployment_name,
    credential = credential
)

await client.configure_azure_monitor(
    enable_sensitive_data=True
)

### Define the Sales Agent

In [ ]:
sales_agent = Agent(
    client=client,

    name="Sales-Agent",

    instructions="""
    You are a sales specialist for Contoso Cloud.

    You handle:
    - product questions
    - pricing
    - upgrades
    - downgrades
    - purchasing

    If the user has an existing customer support
    problem such as:

    - billing errors
    - duplicate charges
    - refunds
    - technical problems
    - account problems

    hand off to Support-Agent.

    Do not attempt to resolve support problems yourself.
    """,

    require_per_service_call_history_persistence=True
)

### Define the Support Agent

In [ ]:
support_agent = Agent(
    client=client,

    name="Support-Agent",

    instructions="""
    You are a customer support specialist
    for Contoso Cloud.

    You handle:

    - billing problems
    - duplicate charges
    - refunds
    - account issues
    - technical support

    When you receive a support request,
    provide the best possible response.

    Do not claim that you performed actions,
    accessed the customer's account, initiated
    refunds, or contacted internal teams unless
    you actually have tools that allow you to
    perform those actions.

    If the user asks about:

    - buying a product
    - pricing
    - upgrades
    - downgrades

    hand off to Sales-Agent.
    """,

    require_per_service_call_history_persistence=True
)

### Build the Handoff Workflow

In [ ]:
from agent_framework import WorkflowBuilder
from agent_framework.orchestrations import HandoffBuilder


workflow = (
    HandoffBuilder(
        name="customer_support_handoff",

        participants=[
            sales_agent,
            support_agent
        ]
    )

    .with_start_agent(
        sales_agent
    )

    .build()
)

### Run a Simple Sales Query

In [ ]:
request = """
Can you please help me with your product pricing plans?
"""

In [ ]:
async for event in workflow.run(
    request,
    stream=True
):

    if event.type == "handoff_sent":

        print(
            f"\n\n HANDOFF: "
            f"{event.data.source} "
            f"→ {event.data.target}\n"
        )

    elif event.type == "output":

        data = event.data

        if hasattr(data, "text") and data.text:

            print(
                data.text,
                end="",
                flush=True
            )

### Test a Technical Request

In [ ]:
request = """
My application cannot authenticate with
Contoso Cloud.

Every API request returns:

401 Unauthorized

I regenerated the API key but the problem
still occurs.

Can you help?
"""

In [ ]:
async for event in workflow.run(
    request,
    stream=True
):

    if event.type == "handoff_sent":

        print(
            f"\n\n HANDOFF: "
            f"{event.data.source} "
            f"→ {event.data.target}\n"
        )

    elif event.type == "output":

        data = event.data

        if hasattr(data, "text") and data.text:

            print(
                data.text,
                end="",
                flush=True
            )